In [1]:
%load_ext autoreload
%autoreload 2
import main
import storage
import ranks
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import itertools
from itertools import takewhile,dropwhile
from fractions import Fraction
import pd_cols
import scipy.stats as stats
import scikit_posthocs as sp
from statsmodels.stats.weightstats import ttost_paired
import scipy
import math
import storage, ranks
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import itertools
from itertools import takewhile,dropwhile
from fractions import Fraction
import pd_cols


/home/tzasadil/src/Autoencoders-in-blackbox-optimization/.venv/lib/python3.11/site-packages/cocopp/toolsdivers.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-04-03 22:19:14.305040: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-03 22:19:14.305279: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-03 22:19:14.345454: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical 

Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


/home/tzasadil/src/Autoencoders-in-blackbox-optimization/.venv/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
# df = storage.merge_and_load()
# df=df[~(df['note'] == 'ForcedSpecimen0')]
# storage.overwrite(df)
# 42

In [ ]:
# %matplotlib inline

def p(func, /, *args, **keywords):
    def newfunc(*fargs, **fkeywords):
        newkeywords = {**keywords, **fkeywords}
        return func(*args, *fargs, **newkeywords)
    newfunc.func = func
    newfunc.args = args
    newfunc.keywords = keywords
    return newfunc

if False: #delete the all experiment records
        shutil.rmtree("ppdata")
        shutil.rmtree("exdata")
        shutil.rmtree("graphs")

df_og = storage.merge_and_load()
df_og = df_og[(df_og['note']=='') | (df_og['note']=='none') ]
df_og['full_desc'] = df_og.apply(pd_cols.get_full_desc,axis=1)
df_og = ranks.compute_ranks(df_og)

# helper functions and axilliary data
pd.options.mode.chained_assignment = None          # prevents displaying a useless warning

def df_enhance(df:pd.DataFrame):
        # df['true_evaluations'] = (df['pop_size'] * df['true_ratio']).map(int)
        df['gen_mult'] = df['gen_mult'].map(int)
        # df['dim_red'] = df['dim_red'].replace('','none')
        # del df['dim_red']
        # del df['gen_mult']
        df['model'] = df['model'].replace('','none')
        df['pop_size'] = df['pop_size'].replace('None','none')
        df['actual_pop_size'] = df.apply(lambda r: r['pop_size'] if r['pop_size'] != 'none' else 4 + math.floor(3 * math.log(r['dim'])),axis=1)
        df['rank_evals'] = df.apply(lambda r: np.array(range(5*r['dim'],250*r['dim']+1,5*r['dim'])),axis=1)
        df['model_kind'] = df['model'].map(lambda a: (''.join(takewhile(lambda s: s.isalpha(), a))).lower())
        df['surrogate'] = df['model'].map(lambda a: (''.join(takewhile(lambda s: s.isalnum(), a))).lower())

        # df['spearman_corr'] = df['spearman_corr'].map(np.nan_to_num)
        # df['spearman_pval'] = df['spearman_pval'].map(np.nan_to_num)
        return df

df_og = df_enhance(df_og)
# df_og = df_og[df_og['scale_train']]
pure_mask = df_og['gen_mult'].map(int) == 1
pures = df_og[pure_mask]
# pures2 = df_og[df_og['scale_train']==Tru]

# pca_mask = (df_og['model_kind'] == 'gp')&(df_og['dim_red_kind'] == 'pca')&((df_og['pop_size']==48)|(df_og['pop_size']==64))&(df_og['true_ratio'].map(Fraction)==Fraction(1/8))
# pca_df = df_og[pca_mask]

# keep the special cases separate, it makes all the graphing easier
# df_og = df_og[~(
#        (pure_mask&(df_og['pop_size'] != 'none'))
#     #    |(pca_mask&(df_og['dim_red'] != 'pca0.5'))
# )]




In [4]:

df = df_og.copy()

def auc(df):
    def auc_regret(x, y): return np.trapz(y, x)
    return df.groupby(pd_cols.determining_cols)\
        .apply(lambda d: auc_regret(d['evals'].to_numpy(), d['ranks'].to_numpy()))


def equivalence(df):
    wide = df.pivot_table(
        index=['function', 'instance', 'dim'],
        columns='model',
        values='avg_rank'
    )
    heur = wide['doe82'].to_numpy()
    cma  = wide['none'].to_numpy()

    delta = 0.5 # todo: find best val
    pval, _,_ = ttost_paired(heur, cma, low=-delta,upp=delta)
    print('pval of ttost equivalence test:', pval)
    # big pval => h0 cant be rejected => equivalent
    # small => may or may not be equivalent
equivalence(df)

# run for all data to prove significance
def stat_tests(df, descriptor:str = ''):
    if(descriptor!=''):
        print(descriptor)
    wide = df.pivot_table(
        index=['function', 'instance', 'dim'],
        columns='model',
        values='avg_rank'
    )
    algo_cols = list(wide.columns)
    fried =  stats.friedmanchisquare(*[wide[col].to_numpy() for col in algo_cols])
    print('fried stat:', fried.statistic)
    print('fried p:', fried.pvalue)
    # now post-hoc Nemenyi (or pairwise Wilcoxon with Holm correction). Report Kendall’s W or mean rank.
    ph = sp.posthoc_nemenyi_friedman(wide.to_numpy())
    ph.index = algo_cols
    ph.columns = algo_cols
    print('post hoc nemenyi friedman:\n', ph)
stat_tests(df)

KeyError: 'doe82'

In [ ]:
# df = storage.merge_and_load()
# df['model_kind'] = df['model'].map(lambda a: (''.join(takewhile(lambda s: s.isalpha(), a))).lower())
# # df[df['model_kind']=='doe']['dists'] = [d[-len(c):] for d,c in  zip(df[df['model_kind']=='doe']['dists'].to_list(),df[df['model_kind']=='doe']['correct_invariant'].to_list())]
# df['dists'] = df.apply(lambda r: r['dists'][-len(r['correct_invariant']):] if r['model_kind']=='doe' else r['dists'],axis=1)
# print()
# storage.overwrite(df)

In [ ]:
# import matplotlib.pyplot as plt
def xy_scatter(xs, ys, len_mod_s, len_mod_e, xdesc, ydesc):
    fig, ax = plt.subplots()
    if xs == 'index':
        xs, ys = list(zip(*[(i,item) for arr in  xs.to_list() for (i,item) in enumerate(arr[int(len(arr)*len_mod_s):int(len(arr)*len_mod_e)])]))
    else:
        xs = np.array([item for arr in  xs.to_list() for item in arr[int(len(arr)*len_mod_s):int(len(arr)*len_mod_e)]])
        ys = np.array([item for arr in  ys.to_list() for item in arr[int(len(arr)*len_mod_s):int(len(arr)*len_mod_e)]])


    not_nan = ~(np.isnan(xs)|np.isnan(ys))
    xs, ys=np.array(xs)[not_nan], np.array(ys)[not_nan]
    ax.scatter(xs, ys, marker='.')

    # m, b = np.polyfit(dists, corr_invariant_tracker, 1)
    lr = scipy.stats.linregress(xs, ys)
    xx = np.linspace(np.min(xs), np.max(xs), num=100)
    ax.plot(xx, lr.slope*xx + lr.intercept,color='red')
    txt = [
        "r^2 = {:.3f}".format(lr.rvalue**2),
        'linreg start: {:.3f}'.format(lr.intercept),
        'linreg end: {:.3f}'.format(lr.slope*np.max(xs) + lr.intercept),

    ]
    ax.annotate('\n'.join(txt), (0.8*np.max(xs), 0.8*np.max(ys)))
    ax.set_xlabel(xdesc.title())
    ax.set_ylabel(ydesc.title())
    fig.show()


In [ ]:
def two_layer_tics(ax):
        plt.xticks(rotation=0, size= 'xx-small')
        for tick in ax.xaxis.get_major_ticks()[1::2]:
                tick.set_pad(15)
def save_and_show(name:str, show=True):
        fig = plt.gcf()
        fig.savefig(f'graphs/{name}.png', bbox_inches='tight')
        if show:
                plt.show()
        plt.close(fig)
def latex_tabular(df: pd.Series):
        label = lambda s: ' ' if s is None else str(s).replace('_', ' ')
        s= "\\begin{tabular}{|lc|}\n"  # + " | ".join(["c"] * len(df.columns)) + "}\n"
        s+= "\\hline\n"
        s+= label(df.index.name) + ' & ' + label(df.name) +'\\\\\n'
        s+= "\\hline\n"
        for k, v in df.items():
                s+= f"{k} & {v:0.2f} \\\\\n"
        s+= "\\hline\n"
        s+= "\\end{tabular}"
        return s
def write_latex_table(df: pd.Series, output_path: str, heading: str | None = None):
        from pathlib import Path
        path = Path(output_path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open('w', encoding='utf-8') as handle:
                if heading:
                        handle.write(f"% {heading}\n")
                handle.write(latex_tabular(df))
                handle.write("\n")
def print_latex(df: pd.Series, output_path: str | None = None, heading: str | None = None):
        table = latex_tabular(df)
        if output_path is None:
                print(table)
                return
        write_latex_table(df, output_path, heading)
def np_apply_axis0(fn=None):
        def inner(arr, fn):
                arr = arr.to_list()
                b = np.apply_along_axis(fn,0,arr)
                return list(b)
        return lambda a: inner(a, fn)
def avg_axis0_rugged(arr):
        arr = arr.to_list()
        m_l = max(map(len, arr))
        masks = []
        padded = []
        for a in arr:
                to_pad = m_l-len(a)
                mask = [True]*len(a) + [False]*(to_pad)
                p_arr = np.pad(a,(0, to_pad), constant_values=0)
                padded.append(p_arr)
                masks.append(mask)
        stacked= np.stack(padded, axis=0)
        num_valid_in_each_col = np.sum(np.array(masks),axis=0)
        avg = np.sum(stacked, 0)/num_valid_in_each_col
        return list(avg)

def close_to(series, num):
        return series.map(lambda a: abs(a - num) <= 1e-3)

def get_param_desc_title(df):
    # beginning + range of values
    def title_stringer(beginning, name):
        mn =  df[name].min()
        mx =  df[name].mx()
        return beginning + ' ' + str(mn) + (('-'+str(mx)) if mn !=mx else '')
    # fun, dim, inst merged into title
    title = title_stringer('fun','function') + '; dim ' + ', '.join([str(a) for a in np.unique(df['dim'])]) + title_stringer('; inst','instance')
    return title

def default_groupby(df, columns):
        map_dict = {
                'ranks':np_apply_axis0(np.average),
                'avg_rank':'mean',
                'last_rank':'mean',
                'elapsed_time':'mean',
                'model':'first',
                'model_kind':'first',
                'surrogate':'first',
                'gen_mult':'first',
        }
        for c in columns:
               if c in map_dict:
                      del map_dict[c]
        res = df.groupby(columns).agg(map_dict)
        return res

baselines = default_groupby(pures, ['pop_size'])
baseline_color = '#E04836'
default_color = 'forestgreen'
def bar(df:pd.DataFrame,x_name=None, y_name='avg_rank',  index_mapper = None, y_mapper = None, regr = False, baseline_i=-1, x_ticklabel_mapper=None, print_table= '', title = '', table_path=None):
        if x_name is not None:
            df = default_groupby(df,x_name)
        df = df.sort_index()

        if print_table != '':
                if table_path is None:
                        print(print_table)
                print_latex(df[y_name], output_path=table_path, heading=print_table)
        colors = [default_color for _ in range(len(df))]
        if baseline_i != -1 :
                colors += [baseline_color]
                df.loc[str(len(df))] = baselines.loc[baseline_i]

        x = df.index.to_numpy()
        y = df[y_name]
        fig, ax = plt.subplots()
        positions = np.arange(len(y))
        ax.bar(positions, y.to_numpy(), color=colors)

        ax.set_ylabel('Rank Percentile')
        xn = df.index.name if x_name == None else x_name
        if xn != None: # df.index.name can be None
                xn = xn.replace('_', ' ').title()
                ax.set_xlabel(xn)
        xticklabels = [str(label) for label in x]
        if x_ticklabel_mapper:
               xticklabels = x_ticklabel_mapper(xticklabels)
        if baseline_i != -1 :
                xticklabels[-1] = 'baseline'
                ax.axhline(y=y.iloc[-1], color=baseline_color, linestyle='dotted')
        ax.set_xticks(positions)
        ax.set_xticklabels(xticklabels, size='small')
        if regr:
                xx = np.arange(len(y)-(1 if baseline_i != -1  else 0))
                m, b = np.polyfit(xx, y[:-1] if baseline_i != -1  else y, 1)
                ax.plot(xx, m*xx + b,color='red', alpha=0.5)
        return ax



In [ ]:
gen_lims = [5, 2, 1]
dims = [2,5,10, None]
func_groups = [
    (1,5, 'Separable Functions'),
    (6,9, 'Functions with low or moderate conditioning'),
    (10,14, 'Ill conditioned functions'),
    (15,19, 'Adequately structured multimodal functions'),
    (20,24, 'Weakly structured multimodal functions'),
    (1,24, 'All functions')
]

for frac_eval_limit in gen_lims:
    for dim in dims:
        for (s,e,description) in func_groups:
            df = df_og.copy()
            title = []
            eval_limit = 999

            if description != '':
                title.append(description)
                df = df[(df['function']>=s)&(df['function']<=e)]

            if dim is not None:
                title.append(f'dim {dim}')
                df = df[(df['dim']==dim)]
            else: title.append(f'all dims')

            if frac_eval_limit != 1:
                a = frac_eval_limit
                title.append(f'first 1/{a} evaluations')
                eval_limit = int(250/frac_eval_limit)
            title = ', '.join(title)
            graph_name = title.replace('/', '')
            table_path = f'graphs/{graph_name}.tex'

            # df = df[(df['note']=='')|((df['gen_mult']==1)&(df['pop_size']=='none'))]
            # df['best_per_sett']
            df['improvement_percent'] = df['vals'].apply(lambda a: 100*(1-(a-a[-1])/(a[3]-a[-1])))
            df['convergence_cutoff'] = df['improvement_percent'].apply(lambda a: np.argmin(a>99.99))
            # df = df[df['convergence_cutoff']>5]
            # df['evals'] = df.apply(lambda r: r['evals'][:r['convergence_cutoff']],axis=1)
            # df['vals'] = df.apply(lambda r: r['vals'][:r['convergence_cutoff']],axis=1)


            # df = df[(df['model']=='doe_32_8')&(df['dim']==2)]
            # xy_scatter(df['dists'],df['spearman_corr'], 0, 1, 'latent space euklidean distance', 'spearman rank correlation')
            # xy_scatter(df,'doe_16_4', 0,1, 'dists', 'spearman_pval', 'latent space euklidean distance', 'spearman rank p-value')

            df['reduced_len'] = df.apply(lambda r: (r['evals']<=eval_limit).astype(int).sum(),axis=1)
            df['rank_len'] = df.apply(lambda r: (r['rank_evals']<=eval_limit).astype(int).sum(),axis=1)

            df['vals'] = df.apply(lambda r: r['evals']<=eval_limit,axis=1)
            df['evals'] = df.apply(lambda r: r['evals'][:r['reduced_len']],axis=1)

            df['ranks'] = df.apply(lambda r: r['ranks'][:r['rank_len']],axis=1)
            df['avg_rank'] = df.apply(lambda r: r['ranks'].mean(),axis=1)
            df['last_rank'] = df.apply(lambda r: r['ranks'][-1],axis=1)
            df['rank_evals'] = df.apply(lambda r: r['rank_evals'][:r['rank_len']],axis=1)

            if df.empty:
                continue

            dfg = df.groupby('model_kind').agg({
                'avg_rank': 'mean',
                'last_rank': 'mean',
                'elapsed_time': 'mean',
                'model': 'first',
                'model_kind': 'first',
                'surrogate': 'first',
                'gen_mult': 'first',
            })
            # dfg['avg_rank'] = dfg['ranks'].apply(np.mean)
            # dfg['median_rank'] = dfg['ranks'].apply(np.median)
            ax = bar(dfg,y_name='avg_rank', print_table=title, table_path=table_path)
            plt.title(title)
            save_and_show(graph_name, False)
            # plt.show()
            # for tick in ax.xaxis.get_major_ticks()[1::2]:
            #     tick.set_pad(15)
            # plt.xticks(size= 25)
            # aaaa = avg_axis0_rugged(df['ranks'])
            #

            # for s,e,description in func_groups:
            #     df1 = df[(df['function']>=s)&(df['function']<=e)]
            #     ax = bar(dfg,y_name='last_rank')
            #     for tick in ax.xaxis.get_major_ticks()[1::2]:
            #         tick.set_pad(15)
            #     plt.xticks(size= 5)



/tmp/ipykernel_214539/2222350689.py:38: RuntimeWarning: divide by zero encountered in divide
  df['improvement_percent'] = df['vals'].apply(lambda a: 100*(1-(a-a[-1])/(a[3]-a[-1])))
/tmp/ipykernel_214539/2222350689.py:38: RuntimeWarning: invalid value encountered in divide
  df['improvement_percent'] = df['vals'].apply(lambda a: 100*(1-(a-a[-1])/(a[3]-a[-1])))
/tmp/ipykernel_214539/2222350689.py:38: RuntimeWarning: divide by zero encountered in divide
  df['improvement_percent'] = df['vals'].apply(lambda a: 100*(1-(a-a[-1])/(a[3]-a[-1])))
/tmp/ipykernel_214539/2222350689.py:38: RuntimeWarning: invalid value encountered in divide
  df['improvement_percent'] = df['vals'].apply(lambda a: 100*(1-(a-a[-1])/(a[3]-a[-1])))
/tmp/ipykernel_214539/2222350689.py:38: RuntimeWarning: divide by zero encountered in divide
  df['improvement_percent'] = df['vals'].apply(lambda a: 100*(1-(a-a[-1])/(a[3]-a[-1])))
/tmp/ipykernel_214539/2222350689.py:38: RuntimeWarning: invalid value encountered in divide


In [ ]:
from pathlib import Path

func_group_labels = [
    ('f1-f5', 1, 5, 'Separable Functions'),
    ('f6-f9', 6, 9, 'Functions with low or moderate conditioning'),
    ('f10-f14', 10, 14, 'Ill conditioned functions'),
    ('f15-f19', 15, 19, 'Adequately structured multimodal functions'),
    ('f20-f24', 20, 24, 'Weakly structured multimodal functions'),
]

analysis_output_dir = Path('graphs/avgs')
analysis_output_dir.mkdir(parents=True, exist_ok=True)

def add_func_group(df: pd.DataFrame) -> pd.DataFrame:
    enriched = df.copy()
    enriched['func_group'] = 'Other'
    enriched['func_group_key'] = 'other'
    for key, start, end, label in func_group_labels:
        mask = enriched['function'].between(start, end)
        enriched.loc[mask, 'func_group'] = label
        enriched.loc[mask, 'func_group_key'] = key
    return enriched

def write_dataframe_tabular(df: pd.DataFrame, output_path: Path, column_format: str):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    latex = df.to_latex(index=False, escape=False, float_format=lambda value: f'{value:0.2f}', column_format=column_format)
    lines = latex.splitlines()
    output_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

analysis_df = add_func_group(df_og)
analysis_df = analysis_df[analysis_df['model_kind'].isin(['doe', 'gp', 'nn', 'none'])].copy()
analysis_df['avg_rank'] = analysis_df['ranks'].apply(np.mean)

per_problem = analysis_df.groupby([
    'func_group', 'func_group_key', 'dim', 'function', 'instance', 'model_kind'
], as_index=False).agg(avg_rank=('avg_rank', 'mean'))

group_summary = per_problem.groupby(['func_group', 'func_group_key', 'dim', 'model_kind'], as_index=False).agg(
    avg_rank=('avg_rank', 'mean'),
    problems=('avg_rank', 'size')
)

doe_summary = group_summary[group_summary['model_kind'] == 'doe'].rename(columns={'avg_rank': 'doe_avg_rank'})
baseline_summary = group_summary[group_summary['model_kind'] == 'none'].rename(columns={'avg_rank': 'baseline_avg_rank'})
peer_summary = group_summary[group_summary['model_kind'] != 'doe'].groupby(['func_group', 'func_group_key', 'dim'], as_index=False).agg(
    peer_avg_rank=('avg_rank', 'mean'),
    best_non_doe_avg_rank=('avg_rank', 'min')
)

doe_group_eval = doe_summary.merge(
    baseline_summary[['func_group', 'dim', 'baseline_avg_rank']],
    on=['func_group', 'dim'],
    how='left'
 ).merge(
    peer_summary,
    on=['func_group', 'func_group_key', 'dim'],
    how='left'
 )

rank_within_group = group_summary.copy()
rank_within_group['model_rank'] = rank_within_group.groupby(['func_group', 'dim'])['avg_rank'].rank(method='dense')
doe_rank = rank_within_group[rank_within_group['model_kind'] == 'doe'][['func_group', 'dim', 'model_rank']]
doe_group_eval = doe_group_eval.merge(doe_rank, on=['func_group', 'dim'], how='left')

doe_group_eval['vs_baseline'] = doe_group_eval['baseline_avg_rank'] - doe_group_eval['doe_avg_rank']
doe_group_eval['vs_peer_mean'] = doe_group_eval['peer_avg_rank'] - doe_group_eval['doe_avg_rank']
doe_group_eval['vs_best_non_doe'] = doe_group_eval['best_non_doe_avg_rank'] - doe_group_eval['doe_avg_rank']
doe_group_eval = doe_group_eval.sort_values(['dim', 'func_group_key'])

doe_by_dim = doe_group_eval.groupby('dim', as_index=False).agg(
    doe_avg_rank=('doe_avg_rank', 'mean'),
    vs_baseline=('vs_baseline', 'mean'),
    vs_peer_mean=('vs_peer_mean', 'mean'),
    vs_best_non_doe=('vs_best_non_doe', 'mean'),
    mean_rank=('model_rank', 'mean')
).sort_values('dim')

doe_by_func_group = doe_group_eval.groupby(['func_group', 'func_group_key'], as_index=False).agg(
    doe_avg_rank=('doe_avg_rank', 'mean'),
    vs_baseline=('vs_baseline', 'mean'),
    vs_peer_mean=('vs_peer_mean', 'mean'),
    vs_best_non_doe=('vs_best_non_doe', 'mean'),
    mean_rank=('model_rank', 'mean')
).sort_values('func_group_key')

doe_by_dim_export = doe_by_dim.rename(columns={
    'dim': 'Dimension',
    'doe_avg_rank': 'DOE avg. rank',
    'vs_baseline': 'DOE - baseline',
    'mean_rank': 'DOE rank'
})[['Dimension', 'DOE avg. rank', 'DOE - baseline', 'DOE rank']]

doe_by_func_group_export = doe_by_func_group.rename(columns={
    'func_group': 'Function group',
    'doe_avg_rank': 'DOE avg. rank',
    'vs_baseline': 'DOE - baseline',
    'mean_rank': 'DOE rank'
})[['Function group', 'DOE avg. rank', 'DOE - baseline', 'DOE rank']]

dim_table_path = analysis_output_dir / 'doe_by_dim_summary.tex'
group_table_path = analysis_output_dir / 'doe_by_func_group_summary.tex'
heatmap_path = analysis_output_dir / 'doe_group_heatmaps.png'

write_dataframe_tabular(doe_by_dim_export, dim_table_path, 'rccc')
write_dataframe_tabular(doe_by_func_group_export, group_table_path, 'lccc')

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
sns.heatmap(
    doe_group_eval.pivot(index='func_group', columns='dim', values='vs_baseline').loc[[label for _, _, _, label in func_group_labels]],
    annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=axes[0]
 )
axes[0].set_title('DOE vs baseline none\npositive = DOE better')
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Function group')

sns.heatmap(
    doe_group_eval.pivot(index='func_group', columns='dim', values='model_rank').loc[[label for _, _, _, label in func_group_labels]],
    annot=True, fmt='.1f', cmap='YlGn_r', vmin=1, vmax=4, ax=axes[1]
 )
axes[1].set_title('DOE rank among model kinds\n1 = best')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Function group')

fig.savefig(heatmap_path, bbox_inches='tight')
plt.close(fig)

print(f'Wrote {dim_table_path}')
print(f'Wrote {group_table_path}')
print(f'Wrote {heatmap_path}')
display(doe_by_dim_export)
display(doe_by_func_group_export)

Wrote graphs/avgs/doe_by_dim_summary.tex
Wrote graphs/avgs/doe_by_func_group_summary.tex
Wrote graphs/avgs/doe_group_heatmaps.png


,Dimension,DOE avg. rank,DOE - baseline,DOE rank
0,2,60.22175,3.68950,1.6
1,5,56.01225,-1.32625,1.6
2,10,52.05625,4.13475,1.2


,Function group,DOE avg. rank,DOE - baseline,DOE rank
3,Separable Functions,57.033333,-2.306667,1.333333
2,Ill conditioned functions,59.723333,3.001667,2.000000
0,Adequately structured multimodal functions,54.431667,0.206667,1.666667
4,Weakly structured multimodal functions,57.068333,1.986667,1.000000
1,Functions with low or moderate conditioning,52.227083,7.941667,1.333333


In [ ]:
df = df_og.copy()
df = df[((df['note']=='pop scale, dim scale')|((df['gen_mult']==1))&(df['pop_size']=='none'))&(df['dim']==10)]
ranks.coco_plot(df)

In [ ]:
df = df_og
df = df[(df['approx_dist']!= -1)&(df['dim']== 20)]
print(df['approx_dist'].describe())


KeyError: 'approx_dist'

In [ ]:
df = df_og
ax = bar(df,'full_desc',)
for tick in ax.xaxis.get_major_ticks()[1::2]:
    tick.set_pad(15)
plt.xticks(size= 5)
save_and_show('rrrrr')

True
\begin{tabular}{|lc|}
\hline
full desc & avg rank\\
\hline
None_1_250 & 59.92 \\
None_4_doe_16_4_250 & 48.23 \\
None_4_elm100_250 & 59.10 \\
None_4_gp_250 & 62.29 \\
None_4_nn3_250 & 70.46 \\
\hline
\end{tabular}


 C:\Users\Admin\AppData\Local\Temp\ipykernel_18896\1368612982.py:90: FutureWarning:

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

 C:\Users\Admin\AppData\Local\Temp\ipykernel_18896\1368612982.py:104: UserWarning:set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.


KeyError: 'function'

In [ ]:
df = df_og
df = default_groupby(df, ['true_ratio'])
ax = bar(df,'true_ratio')

save_and_show('rrrrr')

In [ ]:
df = pures
# df = df[df['pop_size']]
ax = bar(df, 'pop_size')
ax.set_xlabel('Population Size')
ax.set_title('Normal Evaluation')

save_and_show('pure')

In [ ]:
# pca
from itertools import takewhile,dropwhile

st =False
df=pca_df[pca_df['scale_train']==st]
df = df[(df['model_kind'] == 'gp')&(df['pop_size']==64)]

df['pca_ratio'] = df['dim_red'].map(lambda a: ''.join(dropwhile(lambda s: s.isalpha(), a)))

# df1 = df_og[(df_og['model_kind'] == 'gp')&(df_og['dim_red_kind'] == 'none')&(df_og['pop_size']==48)&(df_og['true_ratio'].map(Fraction)==Fraction(1/8))].iloc[0]
# df1['pca_ratio'] = str(1.0)
# df.loc[str(len(df))] = df1
df1 = df_og[(df_og['model_kind'] == 'gp')&(df_og['dim_red_kind'] == 'none')&(df_og['pop_size']==64)&(df_og['true_ratio'].map(Fraction)==Fraction(1/8))].iloc[0]

df1['pca_ratio'] = str(1.0)
df.loc[str(len(df))] = df1
ax = bar(df,'pca_ratio', regr=True, baseline_i=8)
ax.set_xlabel('pca reduction ratio')
ax.set_title('PCA + GP')

save_and_show('pca')

In [ ]:
df=df_og
df = df[(df['model_kind'] == 'gp')& (df['dim_red_kind'] == 'none')]
df = default_groupby(df, ['true_evaluations', 'pop_size'])
pures2 = pures.set_index(pures['pop_size'].map(lambda n: (n,n)))
df = pd.concat([df, pures2])
ax = bar(df, print_table=False)

save_and_show('pop_evals')

In [ ]:
df=df_og
df = df[(df['model_kind'] == 'gp')]
ax = bar(df, 'dim_red')
save_and_show('gp_dim_red')


In [ ]:
df=df_og
ax = bar(df, 'dim_red_kind', 'elapsed_time')
ax.set_ylabel('Iteration Time (ms)')
save_and_show('elapsed')

In [ ]:
df=df_og
df = df[(df['dim_red_kind'] == 'none')&(df['pop_size']==48)&(df['true_ratio'].map(Fraction)==Fraction(1/8))]
ax = bar(df, 'model')
two_layer_tics(ax)
save_and_show('models')



In [ ]:
# dim_red
df=df_og
ax = bar(df, 'dim_red_kind')
save_and_show('dim_red_kind')


True
\begin{tabular}{|lc|}
\hline
dim red kind & avg rank\\
\hline
none & 60.00 \\
\hline
\end{tabular}


 C:\Users\Admin\AppData\Local\Temp\ipykernel_18896\1368612982.py:90: FutureWarning:

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

 C:\Users\Admin\AppData\Local\Temp\ipykernel_18896\1368612982.py:104: UserWarning:set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.


KeyError: 'function'

In [ ]:
# rank models without nodimred
df=df_og
df1 = df[ (df['true_evaluations']==6) & (df['dim_red_kind'] == 'none')]
ax = bar(df, 'model')
plt.xticks(rotation=90, size= 'small')
save_and_show('models')


In [ ]:
#popsize,true_evals           with only gp
from fractions import Fraction
df=df_og
df=df[(df['model_kind'] == 'gp')&(df['dim_red_kind']=='none')]

ax = bar(df, 'true_ratio',index_mapper = lambda a: Fraction(a))
ax.set_label('dim red, true evals, aux evals')
ax.set_xlabel('truly evaluated fraction of population')
ax.set_ylabel('rank percentile avg')

labels = [item.get_text() for item in ax.get_xticklabels()]
labels[1] = '1/12'
ax.set_xticklabels(labels)
save_and_show('popsize_evals')

KeyError: 'true_ratio'

In [ ]:
#popsize,true_evals           with only gp
df=df_og
df=df[(df['model_kind'] == 'gp')&(df['dim_red_kind']=='none')]
df = default_groupby(df, ['true_evaluations','pop_size'])
ax = bar(df)
ax.set_label('dim red, true evals, aux evals')
ax.set_xlabel('population: (true evaluated, generated)')
ax.set_ylabel('rank percentile avg')
# xlabel = ax.get_xlabel()
# ax.set_xlabel([1,2,3], rotation='horizontal')
# plt.savefig("graphs/pop.png")
# plt.pause(0.01)
plt.show()

NameError: name 'df_og' is not defined

In [ ]:
# df=pca_df[(pca_df['scale_train']==True)&(pca_df['function']==8)&(pca_df['instance']==1)]
df = df_og
df =df[
    (df['dim']==2)&
    (df['function']<=5)

]
ranks.plot_ranks(df, 1)

 c:\Users\Admin\Desktop\Diplomka\ranks.py:160: FutureWarning:Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`


In [ ]:
df = df_og.copy()
df = df[df['note']=='']
out_folders = df['coco_directory'].unique()
np.array(out_folders)[True, ]
ranks.coco_plot(df)

NameError: name 'df_og' is not defined